# 01 - Explore the dataset  (NOTEBOOK)
Looks at the compact dataset `data/processed/reps_T64.npz` that `scripts/build_dataset.py` writes.
Run the cells top to bottom. Kernel: the `C:\squat_venv` environment (see README).

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent                     # works whether Jupyter starts in notebooks/ or squat_error_study/
sys.path.insert(0, str(ROOT / "src"))

npz = ROOT / "data/processed/reps_T64.npz"
d = np.load(npz)
meta = json.load(open(npz.with_suffix(".json")))
print({k: d[k].shape for k in d.files})

## 2. How many reps, recordings and persons per class?

In [ ]:
df = pd.DataFrame({"label": d["label"], "person": d["person"], "recording": d["recording"],
                   "missing_lower": d["missing"], "neck_missing": d["trunk_missing"]})
summary = df.groupby("label").agg(reps=("recording", "size"), recordings=("recording", "nunique"),
                                  persons=("person", "nunique"))
summary

## 3. Known data problems (important for the paper)
* Pose2Sim wrote joints it could not triangulate as exact zeros; the builder turns them into missing values.
* The Neck (and often Head / shoulders) is missing for the whole recording in some recordings, so the
  pipeline's trunk angles are unreliable there. `configs/default.yaml -> data.max_trunk_missing` controls
  whether those reps are kept.

In [ ]:
bad = df[df.neck_missing > 0.5]
print(f"reps with the Neck missing in >50% of frames: {len(bad)} of {len(df)}  "
      f"({bad.recording.nunique()} recordings, persons {sorted(bad.person.unique())})")
bad.groupby('label').size()

## 4. Reps per recording (9, 10, 11, 12 ... are all kept; each recording is weighted equally in training)

In [ ]:
df.groupby("recording").size().value_counts().sort_index().plot.bar(title="reps per recording")
plt.xlabel("number of reps"); plt.ylabel("recordings"); plt.show()

## 5. What does a rep look like? Mean knee-flexion curve per class

In [ ]:
names = meta["frame_feature_names"]
i = names.index("knee_flexion_L")
t = np.linspace(0, 1, d["frame"].shape[1])
for lab in sorted(set(d["label"])):
    plt.plot(t, d["frame"][d["label"] == lab][:, :, i].mean(0), label=lab)
plt.xlabel("normalised rep time"); plt.ylabel("knee flexion L (deg)"); plt.legend(fontsize=7)
plt.title("mean knee flexion per class (resampled to 64 frames)"); plt.show()

## 6. Tempo numbers (resampling hides speed, so these are given to the models separately)

In [ ]:
tp = pd.DataFrame(d["tempo"], columns=meta["tempo_names"]); tp["label"] = d["label"]
tp.groupby("label").mean().round(2)